# 📖 Beyond the Cover Analytics: Inteligência Editorial com NLP & LLMs
## Desafio Técnico & Negocial A3Data para Editora
---
### Contexto do Desafio:
A editora contratou a **A3Data** para explorar uma base histórica com mais de **2 milhões de avaliações de livros** e **207.000 títulos únicos catalogados**.
- **Gargalo Atual**: 5 analistas dedicados em tempo integral, levando cerca de 3 dias úteis por análise exploratória manual (120 horas-homem).
- **Custo Salarial da Equipe**: R$ 5.000,00 por analista = **R$ 25.000,00/mês** (R$ 300.000,00/ano).
- **Objetivos Centrais**:
  1. Compreender a performance de autores (quantitativa, qualitativamente e ao longo do tempo).
  2. Mapear tendências de gêneros e séries temporais históricas.
  3. Identificar leitores ideais para entrevistas qualitativas através de 4 dimensões analíticas rigorosas.
  4. Simulação de Conselho Editorial com consenso estratégico entre Personas Literárias e a Diretoria Executiva/Financeira.
  5. Demonstrar redução de custos e ROI expressivo.

In [ ]:
# 1. Configuração do ambiente e imports
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona o diretório src ao path
sys.path.insert(0, os.path.abspath('../src'))
from data_loader import DataLoader
from nlp_engine import NLPEngine
from llm_engine import LLMEngine
from business_metrics import ROICalculator

sns.set_theme(style='whitegrid')
print('✅ Ambiente analítico configurado com sucesso!')

### 2. Visão Macro do Catálogo e Estatísticas Gerais (EDA)
Consultamos a base analítica consolidada em DuckDB.

In [ ]:
loader = DataLoader(data_dir='../data')
kpis = loader.get_metricas_gerais()

print(f'📚 Total de Títulos Catalogados: {kpis["total_livros"]:,}')
print(f'⭐ Total de Avaliações Consolidadas: {kpis["total_avaliacoes"]:,}')
print(f'👤 Total de Leitores Ativos: {kpis["total_usuarios"]:,}')
print(f'⭐ Nota Média Global: {kpis["media_score_global"]:.2f} / 5.0')

### 3. Tendências Temporais e Distribuição de Avaliações
Analisamos o comportamento do volume de leituras e as notas ao longo dos anos.

In [ ]:
df_tempo = loader.get_eda_tempo_geral()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_tempo, x='ano', y='total_reviews', color='#3B82F6', ax=ax1)
ax1.set_title('Volume Anual de Avaliações Registradas (1996 a 2013)')
ax1.set_xlabel('Ano')
ax1.set_ylabel('Quantidade de Resenhas')
ax1.tick_params(axis='x', rotation=45)

sns.lineplot(data=df_tempo, x='ano', y='score_medio', marker='o', color='#10B981', ax=ax2)
ax2.set_title('Evolução da Nota Média Geral por Ano')
ax2.set_xlabel('Ano')
ax2.set_ylabel('Nota Média (1 a 5)')
ax2.set_ylim(3.8, 4.6)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4. Concentração de Notas e Top Gêneros no Acervo

In [ ]:
df_estrelas = loader.get_distribuicao_estrelas()
df_top_gen = loader.get_top_generos_geral(limit=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_estrelas, x='Estrelas', y='Total Avaliações', palette='Blues_r', ax=ax1)
ax1.set_title('Distribuição de Avaliações por Estrelas (1★ a 5★)')
ax1.set_xlabel('Estrelas')
ax1.set_ylabel('Volume de Resenhas')

sns.barplot(data=df_top_gen, x='Total Avaliações', y='Gênero', palette='viridis', ax=ax2)
ax2.set_title('Top 10 Gêneros por Engajamento de Leitores')
ax2.set_xlabel('Total de Avaliações')

plt.tight_layout()
plt.show()

### 5. Performance por Autor: Melhores vs Menos Bem Avaliados
Identificação dos autores mais bem avaliados e com maior atrito (mínimo de 40 resenhas).

In [ ]:
df_melhores, df_piores = loader.get_ranking_geral_autores(min_avaliacoes=40, limit=8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=df_melhores, x='Nota Média', y='Autor', palette='Greens_r', ax=ax1)
ax1.set_title('Top 8 Autores Mais Bem Avaliados (Média ⭐)')
ax1.set_xlim(4.0, 5.0)

sns.barplot(data=df_piores, x='Nota Média', y='Autor', palette='Reds', ax=ax2)
ax2.set_title('Top 8 Autores com Menor Nota Média (Média ⭐)')
ax2.set_xlim(1.0, 4.0)

plt.tight_layout()
plt.show()

### 6. Processamento de Linguagem Natural (NLP) e Séries Temporais do Autor
Análise detalhada de um autor específico (exemplo: J. R. R. Tolkien) com NLP de Positiva, Neutra e Negativa.

In [ ]:
autor_foco = 'J. R. R. Tolkien'
df_autor_reviews = loader.get_todas_reviews_autor(autor_foco)
nlp = NLPEngine()
metricas_autor = nlp.calcular_metricas_autor(df_autor_reviews)

print(f'Autor em Análise: {autor_foco}')
print(f'Total de Resenhas: {metricas_autor["total_avaliacoes"]:,}')
print(f'Nota Média: ⭐ {metricas_autor["nota_media"]:.2f} / 5.0')
print(f'🟢 Resenhas Positivas: {metricas_autor["pct_positivas"]:.1f}% ({metricas_autor["total_positivas"]:,})')
print(f'🟡 Resenhas Neutras: {metricas_autor["pct_neutras"]:.1f}% ({metricas_autor["total_neutras"]:,})')
print(f'🔴 Resenhas Críticas: {metricas_autor["pct_negativas"]:.1f}% ({metricas_autor["total_negativas"]:,})')
print(f'Principais termos positivos: {metricas_autor["top_termos_elogio"]}')
print(f'Principais termos críticos: {metricas_autor["top_termos_critica"]}')

### 7. Radar de Entrevistas: Scoreboard dos 4 Pilares Analíticos
Leitores elegíveis (>= 10 reviews, >= 5 livros) avaliados em 4 dimensões (Perspicácia 35%, Pérolas 25%, Tendências 20%, Versatilidade 20%).

In [ ]:
df_scoreboard = loader.get_scoreboard_entrevistas(top_n=10)
print('Top Candidatos Qualificados para Entrevista em Profundidade:')
display(df_scoreboard[[
    'profileName', 'indice_ajustado', 'n_reviews', 'n_livros',
    'perspicacia_pct', 'perolas_pct', 'tendencias_pct', 'versatilidade_pct', 'especialista_pico'
]])

### 8. Simulação Estratégica: Mesa-Redonda com as 3 Personas & Diretoria Editorial
Debate aprofundado entre Marta Albuquerque, Diego Ramos, Helena Souza, CFO e CEO com consenso formalizado.

In [ ]:
llm = LLMEngine()
debate = llm.simular_debate_conselho_editorial(autor_foco, df_autor_reviews)

print('--- FALAS DO CONSELHO EDITORIAL ---')
for f in debate['dialogo']:
    print(f"{f['personagem']} ({f['papel']}): {f['fala']}\n")

print('--- CONSENSO UNIFICADO DA DIRETORIA ---')
for k, v in debate['consenso'].items():
    print(f'{k.upper()}: {v}')

### 9. Business Case & Análise de Retorno sobre Investimento (ROI)
Comparativo financeiro entre o modelo manual de 5 analistas e a plataforma automatizada.

In [ ]:
calc = ROICalculator()
relatorio = calc.gerar_relatorio_executivo()

print(f"Custo Anual Atual (5 Analistas): R$ {relatorio['cenario_atual']['custo_total_anual_reais']:,.2f}")
print(f"Custo Anual com a Plataforma: R$ {relatorio['cenario_proposto']['custo_total_anual_reais']:,.2f}")
print(f"Economia Anual Líquida: R$ {relatorio['roi_e_ganhos']['economia_anual_liquida_reais']:,.2f}")
print(f"Tempo por Análise: de 120 horas para {relatorio['cenario_proposto']['tempo_por_analise_horas']} horas (redução de 99%)")
print(f"ROI no 1º Ano: {relatorio['roi_e_ganhos']['roi_percentual']:.1f}%")